In [ ]:
# Setup: installs MAX and this book's `minillm` package with uv if they are
# missing, as on Colab or Kaggle (turn on Internet in Kaggle's settings
# first). In the book's pixi environment, this cell does nothing.
import importlib.util
import os
import shutil
import subprocess
import sys

BOOK_REF = "main"  # Branch or tag of github.com/modular/max-llm-book.
BOOK_SRC = os.environ.get(
    "MAX_LLM_BOOK_SRC",
    f"git+https://github.com/modular/max-llm-book@{BOOK_REF}",
)


def uv_install(*packages: str) -> None:
    if shutil.which("uv") is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "uv"], check=True
        )
    subprocess.run(
        [
            "uv",
            "pip",
            "install",
            "--python",
            sys.executable,
            "-q",
            "--prerelease",
            "allow",
            "--index-strategy",
            "unsafe-best-match",
            "--extra-index-url",
            "https://whl.modular.com/nightly/simple/",
            *packages,
        ],
        check=True,
    )


if importlib.util.find_spec("max") is None:
    uv_install("max[all]")
if importlib.util.find_spec("minillm") is None:
    uv_install(BOOK_SRC)
# Cells run `max` and `mojo` as commands; find them next to this Python even
# when the environment isn't activated.
os.environ["PATH"] = (
    os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
)
# MAX's matmul needs an NVIDIA GPU from the Ampere generation (compute
# capability 8.0) or newer. Hide older ones, such as Colab's T4, so MAX uses
# the CPU instead of failing.
if shutil.which("nvidia-smi"):
    caps = subprocess.run(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    ).stdout.split()
    usable = [str(i) for i, cap in enumerate(caps) if float(cap) >= 8.0]
    if len(usable) < len(caps):
        os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(usable)
        print(
            f"Using {len(usable)} of {len(caps)} GPUs; MAX needs compute capability 8.0 or newer."
        )

# 6.1 Qwen3: normalized queries and keys

Most open models are close relatives of the Llama architecture you built in
[Chapter 1](https://llm.modular.com/ch01.html). To bring up a new one, you find
what changed and why. This section adds Qwen3, here the 0.6-billion-parameter
model `Qwen/Qwen3-0.6B`. It subclasses your Llama classes, overrides only what
differs, and verifies the result with the tools from
[Chapter 4](https://llm.modular.com/ch04.html).

In [ ]:
import logging
import warnings

import numpy as np
import plotly.graph_objects as go
import torch
from max.driver import CPU
from max.dtype import DType
from max.pipelines import AutoTokenizer
from max.tensor import Tensor
from max.tensor.tensor import TensorType
from transformers import AutoModelForCausalLM

from minillm import viz
from minillm.device import pick_device
from minillm.config import MODEL_ID, LlamaConfig
from minillm.layers import apply_rope, rope_cos_sin
from minillm.model import load_model
from minillm.sampling import generate
from minillm.testing import compare_outputs
from minillm.weights import load_checkpoint

logging.getLogger("max").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="The eager interpreter failed")
DEVICE = pick_device()  # a GPU if MAX finds one, otherwise the CPU
QWEN3 = "Qwen/Qwen3-0.6B"

## What changed

Compare the two configs:

In [ ]:
smol, qwen = (
    LlamaConfig.from_pretrained(MODEL_ID),
    LlamaConfig.from_pretrained(QWEN3),
)
for field in [
    "vocab_size",
    "hidden_size",
    "num_hidden_layers",
    "num_attention_heads",
    "num_key_value_heads",
    "head_dim",
    "intermediate_size",
    "rope_theta",
    "rms_norm_eps",
]:
    if field == "vocab_size":
        print(f"{'':22} {'SmolLM2':>10} {'Qwen3':>10}")
    print(f"{field:22} {getattr(smol, field):>10} {getattr(qwen, field):>10}")

Four differences matter:

- A larger vocabulary, 151,936 tokens, which covers many languages and code. The
  embedding alone holds 156M parameters, about a quarter of the model's 596M.
- An explicit `head_dim`: 16 heads of 128 dimensions make 2048, *wider* than the
  1024-dimension hidden state. `q_proj` projects up, and `o_proj` projects back
  down. The config class from
  [Section 1.3](https://llm.modular.com/01_03_tokens_and_weights.html) reads
  `head_dim` when a model sets it.
- A larger RoPE base, 1,000,000, so the rotations turn more slowly and positions
  stay distinguishable over longer contexts.
- QK-norm: an RMSNorm on every query and key head, before RoPE.

The first three are config values that your code already handles. QK-norm
needs new code.

## QK-norm

Attention scores are dot products of queries and keys. During training,
nothing stops some heads' queries and keys from growing until a few scores
dominate every softmax, which can destabilize the training of large models.
Normalizing the queries and keys, with a learned scale per dimension, bounds
the scores. Qwen3 and Gemma 3 normalize each head separately; OLMo 2
normalizes the whole query and key projections.

For each head's query $q$ (and likewise each key), with $d$ = `head_dim` and
a learned weight $w$ of size $d$:

$$\hat{q} = \frac{q}{\sqrt{\tfrac{1}{d}\sum_j q_j^2 + \epsilon}} \odot w$$

The division gives every query the same root mean square, so the length of
$\hat q$ depends on $w$ and on $q$'s direction, not on $q$'s size. A score
$\hat q \cdot \hat k$ is then bounded however large $q$ and $k$ get. In code,
QK-norm is two `RMSNorm`s of size `head_dim` inside attention, applied to
each head after the projection and before RoPE. The other classes only swap
in the new attention:

In [ ]:
# export: minillm/qwen3.py
from __future__ import annotations

from max.graph import Dim
from max.nn import ModuleList, RMSNorm
from max.tensor import Tensor

from minillm.attention import Attention
from minillm.config import LlamaConfig
from minillm.layers import apply_rope, rope_cos_sin
from minillm.model import DecoderLayer, LlamaForCausalLM, LlamaModel


class Qwen3Attention(Attention):
    """Llama attention with an RMSNorm on each query and key head."""

    def __init__(self, config: LlamaConfig) -> None:
        super().__init__(config)
        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)

    def project(
        self, x: Tensor, start: int | Dim | Tensor
    ) -> tuple[Tensor, Tensor, Tensor]:
        batch, seq, _ = x.shape
        q = self.q_proj(x).reshape([batch, seq, self.num_heads, self.head_dim])
        k = self.k_proj(x).reshape(
            [batch, seq, self.num_kv_heads, self.head_dim]
        )
        v = self.v_proj(x).reshape(
            [batch, seq, self.num_kv_heads, self.head_dim]
        )
        q, k = self.q_norm(q), self.k_norm(k)
        cos, sin = rope_cos_sin(
            self.head_dim, self.rope_theta, start, seq, x.mesh
        )
        return apply_rope(q, cos, sin), apply_rope(k, cos, sin), v


class Qwen3DecoderLayer(DecoderLayer):
    def __init__(self, config: LlamaConfig) -> None:
        super().__init__(config)
        self.self_attn = Qwen3Attention(config)


class Qwen3Model(LlamaModel):
    def __init__(self, config: LlamaConfig) -> None:
        super().__init__(config)
        self.layers = ModuleList(
            [Qwen3DecoderLayer(config) for _ in range(config.num_hidden_layers)]
        )


class Qwen3ForCausalLM(LlamaForCausalLM):
    """Qwen3: a Llama model whose attention normalizes queries and keys."""

    def __init__(self, config: LlamaConfig) -> None:
        super().__init__(config)
        self.model = Qwen3Model(config)

The new parameters are named `q_norm` and `k_norm`, as in Qwen3's checkpoint, so
`load_model` from [Section 1.6](https://llm.modular.com/01_06_full_model.html)
loads it unchanged. `load_model` builds the model inside `F.lazy()`, so the
Llama layers that the subclasses replace are never allocated.

In [ ]:
model = load_model(qwen, load_checkpoint(QWEN3), DEVICE, Qwen3ForCausalLM)
compiled = model.compile(
    TensorType(DType.int64, ("batch", "seq_len"), device=DEVICE)
)
n_params = sum(p.num_elements() for _, p in model.parameters)
print(
    f"{n_params / 1e6:.0f}M values, including the untied copy of the embedding"
)

To see what QK-norm guarantees on real activations, run layer 0's attention on
a sentence: its queries and keys, with RoPE, the scores $q \cdot k / \sqrt{d}$,
and their softmax. Then scale the layer's input up, which scales queries and
keys alike, as projection weights that grew during training would. The
*entropy* of a softmax row measures how spread out a query's attention is: 0
when one key takes all the weight, $\log n$ when $n$ keys share it equally.

In [ ]:
attn = model.model.layers[0].self_attn
ids = AutoTokenizer(QWEN3)(
    ["Normalizing queries and keys keeps attention scores in a stable range."]
)["input_ids"][0]
seq = len(ids)
h = model.model.layers[0].input_layernorm(
    model.model.embed_tokens(Tensor(np.array([ids]), device=DEVICE))
)
cos, sin = rope_cos_sin(attn.head_dim, attn.rope_theta, 0, seq, DEVICE)
causal = np.triu(np.full((seq, seq), -np.inf), 1)


def softmax(z: np.ndarray) -> np.ndarray:
    e = np.exp(z.astype(np.float64) - z.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)


def entropy_per_head(scale: float, qk_norm: bool) -> np.ndarray:
    """Mean attention entropy of each query head, in nats."""
    x = h * scale
    q = attn.q_proj(x).reshape([1, seq, attn.num_heads, attn.head_dim])
    k = attn.k_proj(x).reshape([1, seq, attn.num_kv_heads, attn.head_dim])
    if qk_norm:
        q, k = attn.q_norm(q), attn.k_norm(k)
    q = apply_rope(q, cos, sin).to_numpy()[0]
    k = apply_rope(k, cos, sin).to_numpy()[0]
    k = np.repeat(k, attn.num_heads // attn.num_kv_heads, axis=1)
    scores = np.einsum("qhd,khd->hqk", q, k) / np.sqrt(attn.head_dim)
    p = softmax(scores + causal)
    # Position 0 sees only itself, so its entropy is always 0.
    return -(p * np.log(p + 1e-30)).sum(-1)[:, 1:].mean(-1)


entropy = {
    (scale, qk_norm): entropy_per_head(scale, qk_norm)
    for scale in [1, 3, 10]
    for qk_norm in [False, True]
}
for (scale, qk_norm), e in entropy.items():
    label = "with" if qk_norm else "without"
    print(
        f"input x{scale:<3} {label:7} QK-norm: mean entropy {e.mean():.2f} nats"
    )

In [ ]:
steps = [
    [
        go.Bar(
            x=np.arange(attn.num_heads),
            y=entropy[scale, qk_norm],
            name=f"{'after' if qk_norm else 'before'} QK-norm",
            marker_color=viz.BLUE if qk_norm else viz.ORANGE,
            hovertemplate="head %{x}: %{y:.2f} nats<extra>%{fullData.name}</extra>",
        )
        for qk_norm in [False, True]
    ]
    for scale in [1, 3, 10]
]
fig = go.Figure(
    layout=dict(
        title="Attention entropy per head, Qwen3 layer 0",
        height=460,
        barmode="group",
        xaxis=dict(title="query head", dtick=1),
        yaxis=dict(
            title="mean entropy (nats)",
            range=[0, 1.1 * max(e.max() for e in entropy.values())],
        ),
    )
)
viz.show(viz.with_slider(fig, steps, ["x1", "x3", "x10"], "input scaled "))

Move the slider. Without the norm (orange), the attention depends on the scale.
Scaling the input by 10 makes queries and keys 10 times longer and the scores
100 times larger, and the softmax collapses: the orange bars fall close to 0,
so each query attends to almost a single key. At x1 the orange bars sit higher
than the blue ones, because Qwen3's raw queries and keys are small; the trained
model always normalizes them. The blue bars don't move at all. After `q_norm` and `k_norm`, a query's or key's length depends
only on its direction and the learned per-dimension scale, so no input can make
a score blow up.

## Verify it

Compare with Hugging Face's PyTorch Qwen3 on a few prompts, using the metrics
from [Section 4.2](https://llm.modular.com/04_02_logit_verification.html):

In [ ]:
tokenizer = AutoTokenizer(QWEN3)
reference = AutoModelForCausalLM.from_pretrained(
    QWEN3, torch_dtype=torch.float32
).eval()


prompts = {
    "english": "The capital of France is",
    "code": "def add(a, b):\n    return",
    "french": "Le chat est assis sur",
}
for name, text in prompts.items():
    ids = tokenizer([text])["input_ids"][0]
    ours = compiled(Tensor(np.array([ids]), device=DEVICE)).to_numpy()[0]
    with torch.no_grad():
        theirs = reference(torch.tensor([ids])).logits[0].numpy()
    r = compare_outputs(ours, theirs, atol=1e-3, rtol=1e-3)
    p, q = softmax(theirs), softmax(ours)
    kl = float((p * (np.log(p + 1e-30) - np.log(q + 1e-30))).sum(-1).mean())
    top1 = (ours.argmax(-1) == theirs.argmax(-1)).mean()
    print(
        f"{name:8} max err {r.max_abs_err:.1e}  KL {kl:.1e}  top-1 {top1:.0%}"
    )
    assert top1 >= 0.99 and kl < 1e-6

In [ ]:
def next_logits(ids: list[int]) -> Tensor:
    return compiled(Tensor(np.array([ids]), device=DEVICE))[0, -1]


prompt = tokenizer(["The three primary colors are"])["input_ids"][0]
print(
    tokenizer.decode(prompt + generate(next_logits, prompt, 24, temperature=0))
)

That is the whole bring-up: about 40 lines of model code for a new architecture,
checked against the reference to rounding error. MAX's own Qwen3 implementation
serves the same model with a paged KV cache and continuous batching, the
techniques that [Chapter 5](https://llm.modular.com/ch05.html) builds:

```bash
max serve --model Qwen/Qwen3-0.6B --quantization-encoding float32 --no-prefer-module-v3
```

`--no-prefer-module-v3` selects MAX's text-generation implementation of
Qwen3. By default MAX prefers an architecture's module-based implementation
([Section 7.1](https://llm.modular.com/07_01_serve_your_model.html) explains
this preference), and while the only module-based one registered under Qwen3's
architecture name is the embedding model, `max serve` picks it for this
checkpoint and fails to start.

Qwen3 is also a *reasoning* model: by default it thinks step by step inside
`<think>` tags before it answers, which costs tokens and improves answers to
hard questions. Its chat template has an `enable_thinking` switch that turns
this off. The handbook's
[choosing the right model](https://handbook.modular.com/getting-started/choosing-the-right-model/)
page discusses that trade-off.

## Exercises

1. Add a KV cache to Qwen3: subclass `CachedAttention` from
   [Section 1.8](https://llm.modular.com/01_08_kv_cache.html) with the same
   `q_norm` and `k_norm`, and check it against the full model.
2. Remove QK-norm: copy `load_model` from
   [Section 1.6](https://llm.modular.com/01_06_full_model.html), build
   `LlamaForCausalLM` instead of `Qwen3ForCausalLM`, and pass `strict=False` to
   `load_state_dict` so that it skips the `q_norm` and `k_norm` weights. Rerun
   the comparison. What happens to the output?
3. Count the parameters in the embedding, the attention projections, and the
   MLPs. How do the proportions compare with SmolLM2's?

## Next

[Section 6.2](https://llm.modular.com/06_02_gemma3.html) adds sliding-window
attention with Gemma 3, and
[Section 6.3](https://llm.modular.com/06_03_mixture_of_experts.html) builds a
mixture of experts.